# GraphRAG Pipeline + Evaluace

Tento notebook obsahuje **dvě části**:

1. **Pipeline** (buňky 1–12) — načtení dokumentu, build grafu, indexy, retrievery, RAG chain
2. **Evaluace** (buňky 13+) — testovací sady, metrikY (BERTScore, ROUGE-L, RAGAS), `evaluate()`

**Důležité:** Evaluace volá přímo retrievery z pipeline (`full_retriever`, `standard_retriever`, `pure_graph_retriever`), takže výsledky jsou vždy identické s produkcí.

---

## Část 1 — Pipeline

In [ ]:
import hashlib
import inspect
import os
import pandas as pd
import pickle
import re
import threading
import time

from dotenv import load_dotenv
from IPython.display import display
from pydantic import BaseModel, Field
from yfiles_jupyter_graphs import GraphWidget

from neo4j import GraphDatabase
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_experimental.llms.ollama_functions import OllamaFunctions
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

load_dotenv()

True

In [ ]:
# Připojení k Neo4j 
graph = Neo4jGraph(
    url=os.environ.get('NEO4J_URI'),
    username=os.environ.get('NEO4J_USERNAME'),
    password=os.environ.get('NEO4J_PASSWORD')
)

# Inicializace modelu 
llm = OllamaFunctions(model="llama3.1:8b", temperature=0, format="json", num_predict=4096)

/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_5993/319638709.py:2: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(
/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_5993/319638709.py:9: LangChainDeprecationWarning: The class `OllamaFunctions` was deprecated in LangChain 0.0.64 and will be removed in 1.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = OllamaFunctions(model="llama3.1:8b", temperature=0, format="json", num_predict=4096)


In [ ]:
file_path = "../documents/md/AF_III_02_01_06_Stroj_Planovani_PA.md"
cache_file = "../cache/AF_III_02_01_06_Stroj_Planovani_PA.pkl"

In [ ]:
with open(file_path, "r", encoding="utf-8") as f:
    md_content = f.read()

# 1. Rozšířené split podle nadpisů (H1 - H6)
headers_to_split_on = [
    ("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3"),
    ("####", "Header 4"), ("#####", "Header 5"), ("######", "Header 6"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(md_content)

# 2. Zkrácení na logické celky
text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=400)
all_documents = text_splitter.split_documents(md_header_splits)

# Odstranění pomocných sekcí (obsah, literatura) — neobsahují znalostní obsah
documents = []
for doc in all_documents:
    h1 = doc.metadata.get("Header 1", "").strip().lower()
    
    # Pokud je hlavní nadpis "obsah" nebo "literatura", chunk zahodit
    if "obsah" in h1 or "zdroje" in h1 or "literatura" in h1:
        continue
        
    documents.append(doc)

print(f"Původně vytvořeno: {len(all_documents)} chunků.")
print(f"Po vyhození Obsahu zbylo k analýze: {len(documents)} čistých chunků.")

Původně vytvořeno: 153 chunků.
Po vyhození Obsahu zbylo k analýze: 150 čistých chunků.


In [ ]:
def get_chunk_hash(text):
    return hashlib.md5(text.encode('utf-8')).hexdigest()

# Výpočet MD5 hashů pro všechny chunky aktuálního dokumentu
for doc in documents:
    doc.metadata["hash"] = get_chunk_hash(doc.page_content)

# Načtení existujících hashů z Neo4j a oprava chybějících záznamů
driver = GraphDatabase.driver(
    os.environ.get('NEO4J_URI'),
    auth=(os.environ.get('NEO4J_USERNAME'), os.environ.get('NEO4J_PASSWORD'))
)

with driver.session() as session:
    result = session.run("MATCH (d:Document) RETURN d.text AS text, d.hash AS hash")
    records = list(result)

    neo4j_hashes = set()
    updates = []

    for record in records:
        if record["hash"]:
            neo4j_hashes.add(record["hash"])
        elif record["text"]:
            # Hash chybí — dopočítá se ze zdrojového textu a zařadí do opravné dávky
            computed_hash = get_chunk_hash(record["text"])
            neo4j_hashes.add(computed_hash)
            updates.append({"text": record["text"], "hash": computed_hash})

    if updates:
        print(f"Oprava chybějících hashů: {len(updates)} dokumentů")
        session.run("""
        UNWIND $updates AS up
        MATCH (d:Document {text: up.text})
        SET d.hash = up.hash
        """, {"updates": updates})

# Odstranění zastaralých chunků z Neo4j (smazané nebo upravené části dokumentu)
current_hashes = {doc.metadata["hash"] for doc in documents}
obsolete_hashes = neo4j_hashes - current_hashes

if obsolete_hashes:
    print(f"Odstraňuji zastaralé chunky z Neo4j: {len(obsolete_hashes)}")
    with driver.session() as session:
        for obs_hash in obsolete_hashes:
            session.run("MATCH (d:Document {hash: $hash}) DETACH DELETE d", {"hash": obs_hash})
        # Odstranění osiřelých entit bez jakékoliv vazby
        session.run("""
        MATCH (e)
        WHERE (e:Hlavni_Pojem OR e:Proces_Cinnost OR e:Nastroj_System OR
               e:Metrika_Parametr OR e:Subjekt_Role OR e:Kapitola OR e:Podkapitola)
          AND NOT (e)--()
        DELETE e
        """)
    neo4j_hashes -= obsolete_hashes
    print("Zastaralé chunky a osiřelé entity odstraněny.")

driver.close()

# Načtení lokální PKL cache
local_cache = {}
if os.path.exists(cache_file):
    with open(cache_file, "rb") as f:
        local_cache = pickle.load(f)

# Synchronizace PKL cache s aktuálním stavem dokumentu
obsolete_pkl_hashes = set(local_cache.keys()) - current_hashes

if obsolete_pkl_hashes:
    print(f"Čištění PKL cache: {len(obsolete_pkl_hashes)} zastaralých záznamů")
    for obs_hash in obsolete_pkl_hashes:
        del local_cache[obs_hash]
    with open(cache_file, "wb") as f:
        pickle.dump(local_cache, f)

# Určení chunků vyžadujících LLM extrakci
missing_in_neo4j = [doc for doc in documents if doc.metadata.get("hash") not in neo4j_hashes]
docs_for_llm = [doc for doc in missing_in_neo4j if doc.metadata.get("hash") not in local_cache]

print(f"Chunky v dokumentu:   {len(documents)}")
print(f"Chunky v Neo4j:       {len(neo4j_hashes)}")
print(f"Chunky v PKL cache:   {len(local_cache)}")
print(f"Chunky ke zpracování: {len(docs_for_llm)}")

# LLM extrakce entit a vztahů — pouze pro chunky chybějící v obou úložištích
if docs_for_llm:
    print(f"Spouštím LLM extrakci: {len(docs_for_llm)} chunků")

    universal_nodes = ["Hlavni_Pojem", "Proces_Cinnost", "Nastroj_System", "Metrika_Parametr", "Subjekt_Role"]
    universal_relationships = ["SKLADA_SE_Z", "VYUZIVA", "MA_VLASTNOST", "PROVADI", "SOUVISI_S"]

    llm_transformer = LLMGraphTransformer(
        llm=llm, allowed_nodes=universal_nodes, allowed_relationships=universal_relationships
    )

    # Iterativní zpracování s průběžným ukládáním cache po každém chunku
    for i, doc in enumerate(docs_for_llm):
        doc_hash = doc.metadata["hash"]
        print(f"  {i+1}/{len(docs_for_llm)} (hash: {doc_hash[:8]})")
        try:
            extracted_graphs = llm_transformer.convert_to_graph_documents([doc])
            if extracted_graphs:
                local_cache[doc_hash] = extracted_graphs[0]
                with open(cache_file, "wb") as f:
                    pickle.dump(local_cache, f)
        except Exception as e:
            print(f"  Chyba u chunku {i+1}, přeskočeno: {e}")
            continue

    print("Extrakce dokončena.")
else:
    if missing_in_neo4j:
        print("Všechna chybějící data načtena z PKL cache, LLM extrakce není vyžadována.")
    else:
        print("Neo4j je aktuální, žádné zpracování není vyžadováno.")

# Příprava grafových dokumentů k nahrání do Neo4j
new_graph_documents = [
    local_cache[doc.metadata["hash"]]
    for doc in missing_in_neo4j
    if doc.metadata.get("hash") in local_cache
]

📊 Celkem chunků v textu: 150
🗄️ Z toho už je v Neo4j: 150
📁 Z toho už máme předpočítáno v PKL: 150
🔥 Bude se přes LLM reálně počítat: 0 chunků
✅ LLM není potřeba, všechna data už jsou v Neo4j.


In [ ]:
# Nahrání extrahovaných grafových dokumentů do Neo4j
if new_graph_documents:
    print(f"Nahrávám {len(new_graph_documents)} dokumentů do Neo4j...")
    graph.add_graph_documents(
        new_graph_documents,
        baseEntityLabel=True,
        include_source=True  # Vytvoří uzel Document propojený s extrahovanými entitami
    )
    print("Nahrání dokončeno.")
else:
    print("Neo4j je aktuální, žádné nové dokumenty k nahrání.")

# Sestavení hierarchie kapitol z metadat chunků
print("Sestavuji stromovou strukturu kapitol...")

driver = GraphDatabase.driver(
    os.environ.get('NEO4J_URI'),
    auth=(os.environ.get('NEO4J_USERNAME'), os.environ.get('NEO4J_PASSWORD'))
)

with driver.session() as session:
    for doc in documents:
        chunk_hash = doc.metadata.get("hash")

        # Sestavení cesty nadpisů H1–H6 pro daný chunk
        # Složené path_id zajišťuje jedinečnost podkapitol se stejným názvem napříč kapitolami
        path_nodes = []
        current_path_id = ""

        for i in range(1, 7):
            h = doc.metadata.get(f"Header {i}")
            if h:
                current_path_id = f"{current_path_id} | {h}" if current_path_id else h
                path_nodes.append({"id": current_path_id, "name": h, "level": i})

        if not path_nodes:
            path_nodes = [{"id": "Obecné", "name": "Obecné", "level": 1}]

        # Vytvoření uzlů Kapitola a propojení do stromové hierarchie
        for i, item in enumerate(path_nodes):
            session.run(
                """
                MERGE (k:Kapitola {id: $id})
                SET k.nazev = $name, k.uroven = $level
                """,
                {"id": item["id"], "name": item["name"], "level": item["level"]}
            )

            # Propojení podkapitoly s nadřazenou kapitolou
            if i > 0:
                parent_item = path_nodes[i - 1]
                session.run(
                    """
                    MATCH (parent:Kapitola {id: $parent_id})
                    MATCH (child:Kapitola {id: $child_id})
                    MERGE (parent)-[:OBSAHUJE_PODKAPITOLU]->(child)
                    """,
                    {"parent_id": parent_item["id"], "child_id": item["id"]}
                )

        # Propojení nejhlubšího uzlu hierarchie s textovým chunkem
        leaf_item = path_nodes[-1]
        session.run(
            """
            MATCH (d:Document {hash: $hash})
            MATCH (k:Kapitola {id: $leaf_id})
            MERGE (k)-[:OBSAHUJE_TEXT]->(d)
            """,
            {"hash": chunk_hash, "leaf_id": leaf_item["id"]}
        )

driver.close()
print("Hierarchie kapitol sestavena.")

✅ Znalosti v Neo4j jsou stoprocentně aktuální, není co přidávat.
⏳ Buduji pevnou stromovou strukturu kapitol z metadat...
✅ Hierarchie kapitol byla úspěšně vygenerována a propojena s texty a entitami!


In [ ]:
# Inicializace embeddingového modelu
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    num_ctx=8192  # Maximální kontextové okno modelu nomic-embed-text
)

# Vytvoření hybridního vektorového indexu nad uzly Document
try:
    vector_index = Neo4jVector.from_existing_graph(
        embeddings,
        search_type="hybrid",
        node_label="Document",
        text_node_properties=["text"],
        embedding_node_property="embedding"
    )
    vector_retriever = vector_index.as_retriever(search_kwargs={"k": 10})
    print("Vektorový index vytvořen.")
except Exception as e:
    print(f"Chyba při vytváření vektorového indexu: {e}")

# Vytvoření fulltextového indexu nad entitami a kapitolami
def create_fulltext_index():
    driver = GraphDatabase.driver(
        os.environ['NEO4J_URI'],
        auth=(os.environ['NEO4J_USERNAME'], os.environ['NEO4J_PASSWORD'])
    )
    with driver.session() as session:
        # Znovuvytvoření indexu — zajišťuje konzistenci při opakovaném spuštění
        session.run("DROP INDEX fulltext_entity_id IF EXISTS")
        session.run("""
        CREATE FULLTEXT INDEX fulltext_entity_id IF NOT EXISTS
        FOR (n:__Entity__|Kapitola|Podkapitola)
        ON EACH [n.id, n.nazev]
        """)
    driver.close()

create_fulltext_index()
print("Fulltextový index nad entitami a kapitolami vytvořen.")

✅ Vektorový index úspěšně vytvořen s max. kontextem.
✅ Fulltextový index pokrývající entity i kapitoly je připraven.


In [ ]:
# Inicializace embeddingového modelu pro manuální indexaci
embeddings_model = OllamaEmbeddings(
    model="nomic-embed-text",
    num_ctx=8192  # Maximální kontextové okno modelu nomic-embed-text
)

driver = GraphDatabase.driver(
    os.environ['NEO4J_URI'],
    auth=(os.environ['NEO4J_USERNAME'], os.environ['NEO4J_PASSWORD'])
)

def build_vector_index():
    with driver.session() as session:
        # Odstranění stávajícího indexu před znovuvytvořením
        session.run("DROP INDEX vector IF EXISTS")

        # Načtení uzlů Document bez embeddingu
        result = session.run(
            "MATCH (d:Document) WHERE d.embedding IS NULL RETURN d.id as id, d.text as text"
        )
        nodes = [{"id": r["id"], "text": r["text"]} for r in result]

        print(f"Uzly ke zpracování: {len(nodes)}")

        for i, node in enumerate(nodes):
            try:
                # Text oříznut na 8 000 znaků
                safe_text = node["text"][:8000]
                vector = embeddings_model.embed_query(safe_text)
                session.run(
                    "MATCH (d:Document {id: $id}) SET d.embedding = $vector",
                    {"id": node["id"], "vector": vector}
                )
                if (i + 1) % 10 == 0:
                    print(f"  Zpracováno {i+1}/{len(nodes)}")
            except Exception as e:
                print(f"  Chyba u uzlu {node['id']}: {e}")

        # Vytvoření vektorového indexu nad existujícími embeddingy
        session.run("""
            CREATE VECTOR INDEX vector IF NOT EXISTS
            FOR (m:Document) ON (m.embedding)
            OPTIONS {indexConfig: {
              `vector.dimensions`: 768,
              `vector.similarity_function`: 'cosine'
            }}
        """)
        print("Vektorový index vytvořen.")

build_vector_index()
driver.close()

🧹 Starý index smazán.
📦 Nalezeno 0 dokumentů v DB. Spouštím individuální embedding...
✅ Vektorový index úspěšně vytvořen manuálně.


In [ ]:
# Dense retriever — kosinová podobnost (baseline Naive RAG)
neo4j_dense_index = Neo4jVector.from_existing_index(
    embeddings,
    index_name="vector",
    search_type="vector",
    node_label="Document",
    text_node_property="text"
)
dense_retriever = neo4j_dense_index.as_retriever(search_kwargs={"k": 20})

# Hybridní retriever — kosinová podobnost + BM25 (Hybrid GraphRAG)
neo4j_hybrid_index = Neo4jVector.from_existing_index(
    embeddings,
    index_name="vector",
    search_type="hybrid",
    keyword_index_name="keyword",
    node_label="Document",
    text_node_property="text"
)
hybrid_retriever = neo4j_hybrid_index.as_retriever(search_kwargs={"k": 20})

print("dense_retriever  — search_type=vector")
print("hybrid_retriever — search_type=hybrid")

✅ dense_retriever  — search_type=vector (Naive RAG)
✅ hybrid_retriever — search_type=hybrid (Hybrid GraphRAG)


In [ ]:
# Schéma pro strukturovaný výstup extrakce entit
class Entities(BaseModel):
    """Informace o klíčových entitách a pojmech v textu."""
    names: list[str] = Field(
        ...,
        description="Všechny klíčové pojmy, technické termíny, procesy a funkce. ZÁSADNÍ: Nesekej víceslovná spojení (např. 'funkce prediktivní analytiky', 'řízení dopravy') na jednotlivá slova!",
    )

# Inicializace modelů
# llm_extraction bez num_predict — parametr je relevantní pouze pro LLMGraphTransformer
# llm_rag s rozšířeným kontextem num_ctx=32768 pro zpracování dlouhého retrieval kontextu
llm_extraction = OllamaFunctions(model="llama3.1:8b", temperature=0, format="json")
llm_rag = ChatOllama(
    model="llama3.1:8b",
    temperature=0,
    num_ctx=32768
)

# Few-shot prompt pro extrakci entit z uživatelského dotazu
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """Jsi expert na extrakci klíčových pojmů z odborných dotazů pro vyhledávání v databázi.
Tvým úkolem je z uživatelské otázky vyextrahovat hlavní entity.

ZÁSADNÍ PRAVIDLO: Odborné pojmy, víceslovné názvy, vlastnosti, metriky nebo procesy ponechávej v celku! Nesekej ustálená spojení na jednotlivá slova.

PŘÍKLADY SPRÁVNÉHO CHOVÁNÍ:
Otázka: "Co je to sekvenční diagram?"
Entity: ["sekvenční diagram"]

Otázka: "Jaké jsou fáze buněčného dělení v biologii?"
Entity: ["fáze buněčného dělení", "biologie"]

Otázka: "Co patří pod funkce prediktivní analytiky v marketingu?"
Entity: ["funkce prediktivní analytiky v marketingu"]

Otázka: "Jaké jsou výhody agilního řízení projektů?"
Entity: ["výhody agilního řízení projektů"]
"""),
    ("human", "{question}")
])

# Extrakční pipeline s validací výstupu přes Pydantic schéma
entity_chain = extraction_prompt | llm_extraction.with_structured_output(Entities)


In [ ]:
def graph_enhanced_retriever(question: str):
    try:
        entities = entity_chain.invoke({"question": question})
        print(f"Extrahované entity: {entities.names}")

        all_documents = []
        seen_docs = set()
        search_queries = []

        relations_text = ""
        seen_relations = set()

        # Sestavení Lucene dotazů z extrahovaných entit
        for entity in entities.names:
            words = [w.lower() for w in entity.split() if len(w) >= 3]
            if words:
                lucene_query = " AND ".join([f"*{w[:5]}*" for w in words])
                search_queries.append(lucene_query)

        # Doplňkový dotaz sestavený přímo ze slov otázky
        question_words = [w.lower() for w in re.findall(r'\w+', question) if len(w) >= 3]
        if question_words:
            direct_query = " OR ".join([f"*{w[:5]}*" for w in question_words])
            search_queries.append(direct_query)

        search_queries = list(set(search_queries))

        for lucene_query in search_queries:
            printed_nodes = set()

            
            response = graph.query(
                """
                CALL db.index.fulltext.queryNodes('fulltext_entity_id', $lucene_query, {limit: 5})
                YIELD node

                // Traversal dolů pouze pro listové uzly bez přímých podkapitol
                OPTIONAL MATCH (node)-[:OBSAHUJE_PODKAPITOLU]->(direct_child)
                WITH node, count(direct_child) AS child_count

                OPTIONAL MATCH (node)-[:OBSAHUJE_PODKAPITOLU*1..5]->(sub_k)
                    WHERE child_count = 0

                // Traversal nahoru zachytí preamble chunky na úrovni H1
                OPTIONAL MATCH (parent_k:Kapitola)-[:OBSAHUJE_PODKAPITOLU*1..3]->(node)

                WITH node,
                     collect(DISTINCT sub_k) + [node] + collect(DISTINCT parent_k) AS all_nodes
                UNWIND all_nodes AS search_node

                MATCH (doc:Document)
                WHERE (doc)-[:MENTIONS]->(search_node) OR (search_node)-[:OBSAHUJE_TEXT]->(doc)

                OPTIONAL MATCH (doc)-[:MENTIONS]->(kapitola_entity:__Entity__)
                WHERE labels(node)[0] = 'Kapitola'

                OPTIONAL MATCH (node)-[r]-(neighbor)
                WHERE type(r) <> 'MENTIONS' AND type(r) <> 'OBSAHUJE_TEXT' AND type(r) <> 'OBSAHUJE_PODKAPITOLU'

                RETURN DISTINCT
                    node.id AS entity_name,
                    labels(node)[0] AS node_type,
                    type(r) AS relation,
                    neighbor.id AS related_entity,
                    kapitola_entity.id AS podrizena_entita,
                    doc.text AS source_doc
                LIMIT 50
                """,
                {"lucene_query": lucene_query}
            )

            for r in response:
                node_name = r.get('entity_name')
                node_type = r.get('node_type', 'neznámý typ')
                rel = r.get('relation')
                neighbor = r.get('related_entity')
                podrizena_entita = r.get('podrizena_entita')

                if node_name not in printed_nodes:
                    print(f"  Nalezen uzel [{node_type}]: '{node_name}'")
                    printed_nodes.add(node_name)

                if rel and neighbor:
                    rel_str = f"Položka '{neighbor}' (vztah ke '{node_name}': {rel})"
                    if rel_str not in seen_relations:
                        relations_text += f"- {rel_str}\n"
                        seen_relations.add(rel_str)

                if podrizena_entita:
                    sub_ent_str = f"Kapitola '{node_name}' obsahuje entitu: '{podrizena_entita}'"
                    if sub_ent_str not in seen_relations:
                        relations_text += f"- {sub_ent_str}\n"
                        seen_relations.add(sub_ent_str)

                doc = r.get('source_doc')
                if doc and doc not in seen_docs and len(doc) > 100:
                    all_documents.append(doc)
                    seen_docs.add(doc)

        print(f"Nalezeno dokumentů z grafu: {len(all_documents)}")
        return relations_text, all_documents

    except Exception as e:
        print(f"Chyba v grafovém retrieveru: {e}")
        return "", []

In [ ]:
# Cross-encoder reranker pro přeřazení dokumentů podle relevance k dotazu
# Použit multilingvální model trénovaný na datasetu mMARCO (včetně češtiny)
try:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
    RERANKER_AVAILABLE = True
    print("Cross-encoder načten: mmarco-mMiniLMv2-L12-H384-v1")
except ImportError:
    RERANKER_AVAILABLE = False
    print("sentence-transformers neni dostupne, reranking preskocen")


def rerank_docs(question: str, docs, top_k: int = 10, label: str = ""):
    if not docs or not RERANKER_AVAILABLE:
        return docs[:top_k]

    # Podpora pro Document objekty i prosté řetězce
    texts = [doc.page_content if hasattr(doc, "page_content") else doc for doc in docs]
    pairs = [(question, text) for text in texts]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)

    print(f"  Reranking {label}({len(docs)} dokumentů → top {top_k}):")
    print(f"  {'Pořadí':<7} {'Skóre':>7}  Náhled")
    print("  " + "-" * 70)
    for rank_i, (score, doc) in enumerate(ranked, 1):
        text = doc.page_content if hasattr(doc, "page_content") else doc
        preview = text[:120].replace("\n", " ")
        marker = "+" if rank_i <= top_k else "-"
        print(f"  {marker} #{rank_i:<3} {score:>7.3f}  {preview}...")
    print()

    return [doc for _, doc in ranked[:top_k]]

✅ Multilingual cross-encoder načten: mmarco-mMiniLMv2-L12-H384-v1
   Podporuje češtinu (trénováno na mMARCO multilingválním datasetu).


In [ ]:
def full_retriever(question: str):
    # Vektorové vyhledávání — hybridní retriever (kosinová podobnost + BM25, k=20)
    vector_docs = hybrid_retriever.invoke(question)
    print(f"Vektorové vyhledávání: {len(vector_docs)} dokumentů")
    print("-" * 60)
    for i, doc in enumerate(vector_docs):
        meta = {k: v for k, v in doc.metadata.items() if k != "hash" and v}
        meta_str = " | ".join(str(v) for v in meta.values()) if meta else "(bez metadat)"
        preview = doc.page_content[:200].replace("\n", " ")
        print(f"  [{i+1}] {meta_str}")
        print(f"        {preview}...")
    print("-" * 60)

    graph_relations_text, graph_docs_list = graph_enhanced_retriever(question)
    print(f"Grafové vyhledávání: {len(graph_docs_list)} dokumentů")
    print("-" * 60)
    for i, doc_text in enumerate(graph_docs_list):
        preview = doc_text[:200].replace("\n", " ")
        print(f"  [{i+1}] {preview}...")
    print("-" * 60)

    reranked_vector = rerank_docs(question, vector_docs, top_k=10, label="[vector] ")
    reranked_graph = rerank_docs(question, graph_docs_list, top_k=10, label="[graph] ") if graph_docs_list else []

    # Výpis finálního pořadí dokumentů předaných LLM
    print("Finální pořadí — vektorové dokumenty:")
    print("  " + "-" * 70)
    for rank_i, doc in enumerate(reranked_vector, 1):
        text = doc.page_content if hasattr(doc, "page_content") else doc
        meta = {k: v for k, v in doc.metadata.items() if k != "hash" and v} if hasattr(doc, "metadata") else {}
        meta_str = " | ".join(str(v) for v in meta.values()) if meta else "(bez metadat)"
        preview = text[:150].replace("\n", " ")
        print(f"  #{rank_i:<3} {meta_str}")
        print(f"        {preview}...")
    print()

    if reranked_graph:
        print("Finální pořadí — grafové dokumenty:")
        print("  " + "-" * 70)
        for rank_i, doc in enumerate(reranked_graph, 1):
            text = doc.page_content if hasattr(doc, "page_content") else doc
            preview = text[:150].replace("\n", " ")
            print(f"  #{rank_i:<3} {preview}...")
        print()

    vector_text = "\n\n".join(
        doc.page_content if hasattr(doc, "page_content") else doc
        for doc in reranked_vector
    )
    graph_text = "\n\n".join(reranked_graph)

    # Ořez kontextu — llama3.1:8b spolehlivě zpracuje přibližně 10 000 znaků
    graph_text_trimmed  = graph_text[:8000]
    vector_text_trimmed = vector_text[:3000]
    graph_rel_trimmed   = graph_relations_text[:1500]

    # Grafové dokumenty jsou primárním zdrojem — řazeny na první pozici v kontextu
    context = (
        f"### 1. DOKUMENTY Z GRAFU (PRIMÁRNÍ ZDROJ - ČTĚTE CELÉ, ODPOVĚĎ JE ZDE):\n{graph_text_trimmed}"
        f"\n\n### 2. VEKTOROVÉ DOKUMENTY (Doplňující textový kontext):\n{vector_text_trimmed}"
        f"\n\n### 3. VZTAHY Z GRAFU (Strukturální kontext):\n{graph_rel_trimmed}"
    )

    # Náhled kontextu odeslaného do LLM
    char_count = len(context)
    sep = "=" * 70
    print(f"Kontext odeslaný do LLM ({char_count:,} znaků):")
    print(sep)
    print("[Sekce 1 — grafové dokumenty]")
    print(graph_text_trimmed[:800].replace("\n\n", "\n") + ("…" if len(graph_text_trimmed) > 800 else ""))
    print()
    print("[Sekce 2 — vektorové dokumenty]")
    print(vector_text_trimmed[:800].replace("\n\n", "\n") + ("…" if len(vector_text_trimmed) > 800 else ""))
    print()
    print("[Sekce 3 — vztahy z grafu]")
    print(graph_rel_trimmed[:800] + ("…" if len(graph_rel_trimmed) > 800 else ""))
    print(sep)

    return context

In [ ]:
template = """

Jsi expertní analytický systém pracující s technickou dokumentací.

### KRITICKÁ PRAVIDLA:
1. ODPOVÍDEJ POUZE NA ZÁKLADĚ DODANÉHO KONTEXTU.
2. Pro výčty použij odrážkový seznam, pro definice odstavce.
3. KOMPLETNÍ VÝČET BEZ ZKRACOVÁNÍ — žádné "a další", "apod.", "atd.".

### KONTEXT:
{context}

### OTÁZKA:
{question}


### ODPOVĚĎ:"""

prompt = ChatPromptTemplate.from_template(template)

# Sestavení RAG chain: retrieval → prompt → LLM → parsování výstupu
chain = (
    {"context": full_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm_rag
    | StrOutputParser()
)

print("RAG chain připraven.")

✅ RAG chain s přísným promptem vytvořen!


### Rychlý test pipeline
Otázku pro ověření, že vše funguje.

In [ ]:
print(chain.invoke("Jaké jsou funkce prediktivní analytiky v plánování nákupu?"))


--- HLEDÁM ODPOVĚĎ NA: 'Jaké jsou funkce prediktivní analytiky v plánování nákupu?' ---

📚 VEKTOROVÉ VYHLEDÁVÁNÍ — nalezeno 20 dokumentů (před rerankingem):
------------------------------------------------------------
  [1] 📂 19. Operativní řízení a plánování výroby
       💬 Účelem kapitoly je vymezit funkcionalitu prediktivní analytiky v rámci operativního řízení a plánování výroby, definovat cílové proměnné a prediktory, formulovat analytické otázky pro konzultace analy...
  [2] 📂 15. Prediktivní analytika obchodu a logistiky
       💬 Účelem kapitoly je vymezit základní funkce prediktivní analytiky v rámci plánování v oblasti marketingu, řízení prodeje, nákupu a dopravy. Pro každou s oblastí jsou definovány cílové proměnné a vybran...
  [3] 📂 14. Prediktivní analytika ve finančním řízení
       💬 Účelem kapitoly  je  vymezit  funkce  prediktivní  analytiky  v rámci  finančního  plánování  a tvorby rozpočtů. Ta je základem pro definování cílových proměnných a jim odpovídajících predi

In [ ]:
def showGraph():
    driver = GraphDatabase.driver(
        uri=os.environ.get('NEO4J_URI'),
        auth=(os.environ.get('NEO4J_USERNAME'), os.environ.get('NEO4J_PASSWORD'))
    )

    with driver.session() as session:
        query = """
        MATCH (s)-[r]->(t)
        RETURN s, r, t
        """
        graph_data = session.run(query).graph()

    driver.close()

    widget = GraphWidget(graph=graph_data)

    def custom_node_label(node):
        props = node.get("properties", {})
        labels = node.get("labels", [])

        # Uzly Document zobrazí začátek textu místo hash identifikátoru
        if "Document" in labels:
            text = props.get("text", "")
            return text[:40] + "..." if text else "Textový blok"

        # Kapitoly a entity zobrazí své id
        return props.get("id", "Neznámý uzel")

    widget.node_label_mapping = custom_node_label
    return widget

showGraph()

---

##  Část 2 — Evaluace

Evaluace **přímo volá retrievery z pipeline** výše — žádná duplicita kódu, žádný rozdíl v odpovědích.

**Retrievery:**
- `Standard RAG` — pouze vektorové vyhledávání (`vector_retriever`, k=20)
- `Pure Graph RAG` — pouze grafové vyhledávání + reranking
- `Hybrid GraphRAG` — kombinace grafu + vektoru + reranking (= `full_retriever`)

**Metriky:**
- `BERTScore` P/R/F1 — sémantická podobnost s ground truth
- `ROUGE-L` P/R/F1 — lexikální překryv s ground truth
- `RAGAS` Faithfulness, AnswerRelevancy, ContextRecall, AnswerCorrectness (LLM judge: Qwen)

In [ ]:
try:
    from bert_score import score as bert_score
    BERTSCORE_AVAILABLE = True
    print("BERTScore načten.")
except ImportError:
    BERTSCORE_AVAILABLE = False
    print("bert-score není dostupný: pip install bert-score")

try:
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True
    print("ROUGE-L načten.")
except ImportError:
    ROUGE_AVAILABLE = False
    print("rouge-score není dostupný: pip install rouge-score")

try:
    from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextRecall
    from ragas import evaluate as evaluate_ragas
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.run_config import RunConfig
    from datasets import Dataset
    RAGAS_AVAILABLE = True
    print("RAGAS načten.")
except ImportError:
    RAGAS_AVAILABLE = False
    print("ragas není dostupný: pip install ragas datasets")

# LLM judge pro RAGAS — nezávislý model oddělený od modelu generujícího odpovědi
llm_judge = ChatOllama(model="qwen2.5:7b", temperature=0, num_ctx=16384)

print("Evaluační závislosti načteny.")
print("LLM judge: qwen2.5:7b")

✅ BERTScore loaded
✅ ROUGE-L loaded
✅ RAGAS loaded
✅ Eval imports hotové
   LLM judge (RAGAS): qwen2.5:7b


/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_5993/1775939859.py:22: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextRecall
/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_5993/1775939859.py:22: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextRecall
/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_5993/1775939859.py:22: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metr

In [ ]:
def naive_rag(question: str) -> str:
    """Baseline — dense retriever s kosinovou podobností, bez rerankingu."""
    docs = dense_retriever.invoke(question)
    return '\n\n'.join(d.page_content for d in docs)


def graph_rag(question: str) -> str:
    """Graph RAG — traversal znalostního grafu s rerankingem."""
    rel, docs = graph_enhanced_retriever(question)
    ranked = rerank_docs(question, docs, top_k=10) if docs else []
    graph_text = '\n\n'.join(ranked)
    return (
        f'### DOKUMENTY Z GRAFU:\n{graph_text[:8000]}'
        f'\n\n### VZTAHY Z GRAFU:\n{rel[:3000]}'
    )


def hybrid_graphrag(question: str) -> str:
    """Hybrid GraphRAG — kombinace traversalu znalostního grafu a hybridního retrieveru."""
    return full_retriever(question)


SCENARE = {
    'Naive RAG':       naive_rag,
    'Graph RAG':       graph_rag,
    'Hybrid GraphRAG': hybrid_graphrag,
}

print("Evaluační scénáře připraveny:")
print("  Naive RAG       — dense retriever, bez rerankingu (baseline)")
print("  Graph RAG       — KG traversal s rerankingem")
print("  Hybrid GraphRAG — KG traversal + hybridní retriever s rerankingem")

✅ Evaluační retrievery připraveny
   Scénáře: ['Naive RAG', 'Graph RAG', 'Hybrid GraphRAG']

Naive RAG       — Dense only, BEZ rerankingu  → baseline (Lewis 2020)
Graph RAG       — KG Retriever + reranking    → přínos grafu
Hybrid GraphRAG — KG + Hybrid + reranking     → hlavní systém


### 📋 Testovací sady

Každá sada je seznam dvojic `(otázka, ground_truth)`.
Pro nový dokument přidej `sada_nazev = [...]`.

In [ ]:
sada_planovani_pa = [
 
    # ── VÝČTOVÉ (3) ────────────────────────────────────────────────────────────
 
    (
        'Jaké jsou funkce prediktivní analytiky v plánování nákupu?',
        '- Clustering: klastry dodavatelů podle dodávaných materiálů, nástrojů, přípravků, '
        'příslušnosti k dodavatelskému řetězci; segmentace dodavatelů podle technickoekonomických '
        'charakteristik a spolehlivosti\n'
        '- Klasifikace: klasifikace dodavatelů do stejnorodých tříd; klasifikace dodávaných '
        'materiálů; klasifikace nakupovaných služeb\n'
        '- Přiřazování podobností: určování podstatných charakteristik dodavatelů a jejich '
        'podobností v rámci definovaných skupin\n'
        '- Profilování: specifikace klíčových charakteristik dodavatelů z pohledu přístupů '
        'k řízení obchodních případů Nákup\n'
        '- Predikce vazeb: vyhodnocování vazeb kvality a včasnosti dodávek materiálů vzhledem '
        'k velikosti a ekonomické síle dodavatelské firmy\n'
        '- Náhodné modelování: hodnocení kvality dodávek jednotlivých dodavatelů vzhledem '
        'k jejich podstatným charakteristikám'
    ),
 
    (
        'Jake jsou dimenze v řešení úloh analytiky přípravy a plánování výrobních zakázek?',
        'Čas - časová dimenze určující dobu poptávky, zahájení, dílčích etap, ukončení výrobní zakázky\n'
        'Regiony - struktura regionů pro specifikaci a realizaci výrobních zakázek\n'
        'Podnikové útvary - sledování a plánování nároků na zdroje podle útvarů firmy\n'
        'Měny - struktura využívaných měn včetně kurzů\n'
        'Nákladové druhy - standardní struktura nákladů v souvislosti s přípravou výrobních zakázek\n'
        'Účetní osnova - struktura účtů pro analýzy nákladů na výrobní zakázky\n'
        'Dodavatelé - dodavatelé materiálů, sestav, přípravků a kooperací\n'
        'Lidské zdroje a mzdy - zaměstnanci, kvalifikační struktura, mzdové složky\n'
        'Sklady - struktura vlastních a pronajatých skladů včetně mezioperačních\n'
        'Materiály - vstupní materiály předmětem normování\n'
        'Materiálové normy, Technologické postupy, Výkonové normy\n'
        'Výrobky - výrobky, polotovary, sestavy předmětem technické přípravy výroby\n'
        'Výrobní střediska - struktura středisek a jejich kapacity\n'
        'Výrobní zakázky - struktura druhů výrobních zakázek'
    ),
 
    (
        'Jake jsou hlavní plánovací dimenze podle jednotlivých skupin?',
        'Základní dimenze: Časová dimenze, Regiony, Odvětví ekonomiky, Měrné jednotky\n'
        'Podniková organizace: Cíle firmy, Podnikové procesy, Činnosti, Podnikové útvary, '
        'Podniková aktiva, Vnitropodnikové zakázky\n'
        'Ekonomické dimenze: Účtová osnova, Účetní období, Kapitálová struktura, Měny, '
        'Nákladové druhy, Druhy cen\n'
        'Externí partneři: Zákazníci, Dodavatelé, Veřejná správa, Finanční ústavy\n'
        'Lidské zdroje a mzdy: Zaměstnanci, Kvalifikační struktura, Věková struktura, '
        'Vzdělávání, Typy školení, Mzdové složky\n'
        'Obchodní dimenze: Zboží, Materiály, Služby, Segmenty trhu, Obchodní kanály\n'
        'Dimenze skladového hospodářství: Sklady, Skladová místa\n'
        'Dimenze majetku: Druhy majetku, Investice, Opravy / údržba\n'
        'Dimenze interní dopravy: Poskytovatelé dopravy, Dopravní prostředky, PHM\n'
        'Dimenze hospodaření s energiemi: Druhy energií, Dodavatelé energií\n'
        'IT služby a zdroje: IT služby, Dodavatelé IT, Aplikace, IT projekty\n'
        'Dimenze řízení výroby: Výrobní zakázky, Výrobky, Výrobní střediska, Výrobní dávky'
    ),
 
    # ── DEFINICE / ÚČEL (3) ────────────────────────────────────────────────────
 
    (
        'Jaké jsou základní cíle plánovacích úloh?',
        '- Vytvořit a využít plánovací systém respektující plánovací metody\n'
        '- Zajistit konsolidaci plánů z různých organizačních jednotek\n'
        '- Zajistit konsolidaci hodnot z různých druhů plánů do výsledného finančního plánu\n'
        '- Zajistit konsolidaci plánů z pohledu různých měn\n'
        '- Automatizovat řízení pracovního toku při přípravě plánu\n'
        '- Efektivně zpřístupňovat sestavené plány zainteresovaným pracovníkům'
    ),
    
    (
        'Jaké plánovací nástroje jsou popsány v dokumentu?',
        'Oracle Enterprise Performance Management, '
        'IBM Planning Analytics, '
        'Anaplan, '
        'Workday, '
        'SAP Analytics Cloud'
    ),
 
    (
        'Jaké jsou cílové proměnné strategického řízení?',
        'Hospodářský výsledek, '
        'Počet plánovaných inovací výroby výrobků a služeb, '
        'Objem nákladů firmy, '
        'MVA (Market Value Added), '
        'EVA (Economic Value Added), '
        'EBITDA, '
        'Počet zákazníků firmy, '
        'Průběžná doba výroby, '
        'Produktivita práce, '
        'Tržní podíl'
    ),
 
    # ── VZTAHOVÉ — vstupy/výstupy a vazby mezi oblastmi (3) ───────────────────
 
    (
        'Jaké analytické otázky řeší plánování prodeje?',
        '- Jak zvýšit úspěšnost a výkonnost byznysu díky vysoké kvalitě plánovacích úloh?\n'
        '- Jak co nejpřesněji a včas zjišťovat budoucí předpokládané obchodní příležitosti?\n'
        '- Jak systematicky sledovat a regulovat stav zásob pro prodejní zakázky?\n'
        '- Jak zajistit dostupnost informací o stavu a předpokládaném vývoji trhu?\n'
        '- Jak správně a racionálně aplikovat plánovací metodiky firmy?\n'
        '- Jak průběžně analyzovat odchylky od vytvořeného plánu prodeje?\n'
        '- Jak nastavit různé možnosti alokace plánovaných hodnot prodejů na útvary?'
    ),
 
    (
        'Jaké jsou dimenze v řešení úloh finančního plánování?',
        'Čas - časová dimenze plánovací horizonty finančních ukazatelů\n'
        'Podnikové procesy - plánování finančních objemů pro zajištění procesů firmy\n'
        'Podnikové útvary - plánování ekonomických charakteristik podle útvarů\n'
        'Střediska - hospodářská, nákladová a zisková střediska\n'
        'Finanční ústavy - řízení pohybů na účtech\n'
        'Měny - struktura využívaných měn a kurzy\n'
        'Nákladové druhy - standardní struktura pro plánování nákladů\n'
        'Účetní období - nepřetržitě po sobě jdoucích dvanáct měsíců\n'
        'Účetní osnova - struktura účtů hlavní knihy a analytického účetnictví\n'
        'Dodavatelé - plánování dodávek podle dodavatelů\n'
        'Zákazníci firmy - plánovaný finanční objem zakázek\n'
        'Zaměstnanci - plánování časových kapacit a utilizace\n'
        'Platební podmínky - způsoby a termíny plateb\n'
        'Obchodní kanály - různé způsoby prodeje'
    ),
 
    (
        'Vyjmenuj analytické otázky ve strategickém plánování.',
        'Navazují roční plány na dlouhodobou strategii a záměry firmy?\n'
        'Jsou všechny vstupy a výstupy jednoznačně oceněny a jsou stanoveny možné výkyvy '
        'v průběhu roku?\n'
        'Jsou stanoveny principy a postupy pro tvorbu strategických plánů?\n'
        'Jsou stanoveny seznamy aktuálně sledovaných klíčových metrik z jednotlivých procesů?\n'
        'Obsahuje plán rezervu na možná rizika?\n'
        'Využívá firma v oblasti strategického řízení metody a nástroje prediktivní analytiky?'
    ),
 
    # ── FAKTICKÉ — jednoduché definice (3) ────────────────────────────────────
 
    (
        'Co jsou osy P-F křivky?',
        'Osa X znázorňuje čas. '
        'Osa Y (svislá) představuje stav zařízení.'
    ),
 
    (
        'Jaké skupiny metrik zahrnuje strategické plánování strojírenské firmy?',
        '- Finanční strategie\n'
        '- Obchodní strategie, strategie cenotvorby, strategie distribuce produktů\n'
        '- Personální strategie a strategie interní i externí komunikace\n'
        '- Marketing a marketingový výzkum\n'
        '- Majetkové a investiční strategie\n'
        '- Informační strategie'
    ),
 
    (
        'Jaké jsou funkce prediktivní analytiky v marketingu?',
        '- Clustering: klastry zákazníků strojírenské firmy podle typu výroby, příslušnosti '
        'k dodavatelskému řetězci, dislokace závodů\n'
        '- Klasifikace: klasifikace zákazníků — rozřazení do stejnorodých tříd\n'
        '- Přiřazování podobností: určování podstatných charakteristik zákazníků a výběr '
        'firem podobných nejlepším zákazníkům\n'
        '- Profilování: specifikace klíčových charakteristik zákazníků z pohledu přístupů '
        'k řízení obchodních případů Prodej\n'
        '- Predikce vazeb: vyhodnocování vazeb marketingových aktivit k dosahovaným '
        'obchodním výsledkům\n'
        '- Náhodné modelování: vyhodnocení kvality promo akcí a vývoj tržeb u vybraných '
        'výrobků a služeb'
    ),
 
]
 
print(f'Celkem otázek: {len(sada_planovani_pa)}')

Celkem otázek: 12


In [ ]:
import threading
import inspect

LLM_TIMEOUT = 180


def compute_bertscore(predictions, references):
    """Vrátí (Precision, Recall, F1) listy pro BERTScore."""
    if not BERTSCORE_AVAILABLE:
        none = [None] * len(predictions)
        return none, none, none
    P, R, F1 = bert_score(
        predictions, references,
        model_type='bert-base-multilingual-cased', lang='cs',
        verbose=False
    )
    return P.tolist(), R.tolist(), F1.tolist()


def compute_rouge(predictions, references):
    """Vrátí (Precision, Recall, F1) listy pro ROUGE-L."""
    if not ROUGE_AVAILABLE:
        none = [None] * len(predictions)
        return none, none, none
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    P, R, F1 = [], [], []
    for pred, ref in zip(predictions, references):
        s = scorer.score(ref, pred)['rougeL']
        P.append(s.precision); R.append(s.recall); F1.append(s.fmeasure)
    return P, R, F1


def call_with_timeout(retriever, llm, question, timeout=LLM_TIMEOUT):
    """Zavolá retriever a LLM s timeoutem. Vrátí (answer, context, elapsed_s)."""
    result = {}

    # Prompt identický s pipeline — zajišťuje konzistenci mezi evaluací a produkcí
    EVAL_PROMPT = """
Jsi expertní analytický systém pracující s technickou dokumentací.

### KRITICKÁ PRAVIDLA:
1. ODPOVÍDEJ POUZE NA ZÁKLADĚ DODANÉHO KONTEXTU.
2. Pro výčty použij odrážkový seznam, pro definice odstavce.
3. KOMPLETNÍ VÝČET BEZ ZKRACOVÁNÍ — žádné "a další", "apod.", "atd.".

### KONTEXT:
{context}

### OTÁZKA:
{question}

### ODPOVĚĎ:"""

    def worker():
        try:
            context = retriever(question)
            prompt  = EVAL_PROMPT.format(context=context, question=question)
            answer  = llm_rag.invoke(prompt).content
            result['ok'] = (answer, context)
        except Exception as e:
            result['error'] = str(e)

    t  = threading.Thread(target=worker)
    t0 = time.time()
    t.start()
    t.join(timeout=timeout)
    elapsed = round(time.time() - t0, 1)

    if t.is_alive():
        return None, None, elapsed
    if 'error' in result:
        return f"ERROR: {result['error']}", '', elapsed
    answer, context = result['ok']
    return answer, context, elapsed


def evaluate(sada, document_name=None, timeout=LLM_TIMEOUT, mode='combined'):
    """
    Spustí evaluaci pro danou testovací sadu.

    Retrievery jsou volány přímo z pipeline — bez duplicity kódu,
    chování je identické se Streamlit aplikací.

    Parametry
    ----------
    sada          : list of (question, ground_truth)
    document_name : str, volitelný — název pro výstupní CSV
    timeout       : int, sekundy na jedno LLM volání (výchozí 180)
    mode          : 'combined' | 'reference' | 'ragas'
        'combined'   — BERTScore + ROUGE-L + RAGAS (výchozí)
        'reference'  — pouze BERTScore + ROUGE-L, bez LLM judge
        'ragas'      — pouze RAGAS
    """
    if not sada:
        print('Testovací sada je prázdná.')
        return None

    if document_name is None:
        frame = inspect.currentframe().f_back
        document_name = next(
            (k for k, v in frame.f_locals.items() if v is sada), 'document'
        ).replace('sada_', '')

    print(f'Evaluace: {document_name}')
    print(f'Otázky: {len(sada)} | Metody: {len(SCENARE)} | '
          f'LLM volání celkem: {len(sada) * len(SCENARE)}')
    print(f'Timeout: {timeout}s | Mode: {mode}\n')

    rows     = []
    out_file = f'../evaluation/evaluation_{document_name}.csv'

    for i, (question, ground_truth) in enumerate(sada, 1):
        print(f'[{i}/{len(sada)}] {question}')
        for method_name, retriever in SCENARE.items():
            print(f'  {method_name}...', end=' ', flush=True)

            answer, context, elapsed = call_with_timeout(retriever, llm_rag, question, timeout)

            if answer is None:
                print(f'Timeout ({elapsed}s), přeskočeno.')
                answer = 'TIMEOUT'; context = ''
            else:
                print(f'{elapsed}s')
                print(f'    {answer[:120].replace(chr(10), " ")}...')

            if answer != 'TIMEOUT' and mode in ('combined', 'reference'):
                bp, br, bf1 = compute_bertscore([answer], [ground_truth])
                rp, rr, rf1 = compute_rouge([answer], [ground_truth])
                bert_p, bert_r, bert_f1 = bp[0], br[0], bf1[0]
                rouge_p, rouge_r, rouge_f1 = rp[0], rr[0], rf1[0]
                print(f'    BERT F1: {round(bert_f1, 3)} | '
                      f'ROUGE-L R: {round(rouge_r, 3)} | '
                      f'ROUGE-L F1: {round(rouge_f1, 3)}')
            else:
                bert_p = bert_r = bert_f1 = None
                rouge_p = rouge_r = rouge_f1 = None

            rows.append({
                'document':     document_name,
                'method':       method_name,
                'question':     question,
                'answer':       answer,
                'context':      context,
                'time_s':       elapsed,
                'ground_truth': ground_truth,
                'BERTScore_P':  bert_p,
                'BERTScore_R':  bert_r,
                'BERTScore_F1': bert_f1,
                'ROUGE_L_P':    rouge_p,
                'ROUGE_L_R':    rouge_r,
                'ROUGE_L_F1':   rouge_f1,
            })

        # Průběžné ukládání po každé otázce
        existing = pd.read_csv(out_file) if os.path.exists(out_file) else pd.DataFrame()
        pd.concat([existing, pd.DataFrame(rows)]).drop_duplicates(
            subset=['method', 'question'], keep='last'
        ).to_csv(out_file, index=False, encoding='utf-8-sig')
        print(f'  Uloženo: {out_file}\n')

    df = pd.DataFrame(rows)

    print('\n' + '=' * 60)
    print('Vygenerované odpovědi')
    print('=' * 60)
    for _, r in df.iterrows():
        print(f'\n[{r["method"]}] {r["question"]}')
        print('-' * 40)
        print(r['answer'])

    df_valid = df[df['answer'] != 'TIMEOUT'].copy()
    if df_valid.empty:
        print('Žádné platné výsledky — všechna volání překročila timeout.')
        return df

    if mode in ('combined', 'ragas') and RAGAS_AVAILABLE:
        print('\nRAGAS evaluace...')
        ragas_llm = LangchainLLMWrapper(llm_judge)
        ragas_emb = LangchainEmbeddingsWrapper(embeddings)

        dataset = Dataset.from_dict({
            'user_input':         df_valid['question'].tolist(),
            'response':           df_valid['answer'].tolist(),
            'retrieved_contexts': [[k] for k in df_valid['context'].tolist()],
            'reference':          df_valid['ground_truth'].tolist(),
        })
        ragas_result = evaluate_ragas(
            dataset=dataset,
            metrics=[Faithfulness(), AnswerRelevancy(), ContextRecall(), AnswerCorrectness()],
            llm=ragas_llm, embeddings=ragas_emb,
            run_config=RunConfig(timeout=300, max_workers=1)
        )
        df_ragas   = ragas_result.to_pandas()
        wanted     = ['faithfulness', 'answer_relevancy', 'context_recall', 'answer_correctness']
        available  = [c for c in wanted if c in df_ragas.columns]
        ragas_cols = df_ragas[available].reset_index(drop=True)
    else:
        ragas_cols = pd.DataFrame({
            'faithfulness':       [None] * len(df_valid),
            'answer_relevancy':   [None] * len(df_valid),
            'context_recall':     [None] * len(df_valid),
            'answer_correctness': [None] * len(df_valid),
        })

    ref_cols = ['BERTScore_P', 'BERTScore_R', 'BERTScore_F1',
                'ROUGE_L_P', 'ROUGE_L_R', 'ROUGE_L_F1']
    df_out = pd.concat([
        df_valid[['document', 'method', 'question', 'answer', 'context',
                  'time_s', 'ground_truth'] + ref_cols].reset_index(drop=True),
        ragas_cols
    ], axis=1)

    metrics_cols = [c for c in ref_cols + ['faithfulness', 'answer_relevancy',
                    'context_recall', 'answer_correctness'] if c in df_out.columns]

    print(f'\nVýsledky per otázka — {document_name}')
    display(df_out[['method', 'question'] + metrics_cols].round(3))

    print(f'\nPrůměrné skóre — {document_name}')
    display(df_out.groupby('method')[metrics_cols].mean().round(3))

    if 'ROUGE_L_R' in df_out.columns and df_out['ROUGE_L_R'].notna().any():
        best_rouge = df_out.groupby('method')['ROUGE_L_R'].mean().idxmax()
        print(f'\nNejlepší ROUGE-L Recall (úplnost):    {best_rouge}')
    if 'BERTScore_F1' in df_out.columns and df_out['BERTScore_F1'].notna().any():
        best_bert = df_out.groupby('method')['BERTScore_F1'].mean().idxmax()
        print(f'Nejlepší BERTScore F1 (sémantika):     {best_bert}')
    if 'answer_correctness' in df_out.columns and df_out['answer_correctness'].notna().any():
        best_ac = df_out.groupby('method')['answer_correctness'].mean().idxmax()
        print(f'Nejlepší Answer Correctness (RAGAS):   {best_ac}')

    existing = pd.read_csv(out_file) if os.path.exists(out_file) else pd.DataFrame()
    pd.concat([existing, df_out]).drop_duplicates(
        subset=['method', 'question'], keep='last'
    ).to_csv(out_file, index=False, encoding='utf-8-sig')
    print(f'\nVýsledky uloženy: {out_file}')
    return df_out


print('evaluate() připravena.')
print('  evaluate(sada_planovani_pa)                     # combined')
print('  evaluate(sada_planovani_pa, mode="reference")   # bez RAGAS')
print('  evaluate(sada_planovani_pa, mode="ragas")       # pouze RAGAS')
print('  evaluate(sada_planovani_pa[:1])                 # jedna otázka')

✅ evaluate() připravena

Použití:
  evaluate(sada_strojirenska_firma)                            # combined
  evaluate(sada_strojirenska_firma, mode="reference")          # bez RAGAS
  evaluate(sada_strojirenska_firma, mode="ragas")              # pouze RAGAS
  evaluate(sada_strojirenska_firma[:1], document_name="q1")    # 1 otázka


###  Spuštění evaluace

In [ ]:
evaluate(
    sada_planovani_pa, 
    document_name='planovani_pa',
    mode='combined',
    timeout=60
)

🚀 Evaluating: planovani_pa
   Otázky: 12 | Metody: 3 | Celkem LLM volání: 36
   Timeout: 60s | Mode: combined

[1/12] Jaké jsou funkce prediktivní analytiky v plánování nákupu?
  ⏳ Naive RAG... ✅ 18.6s
    → Funkce prediktivní analytiky v plánování nákupu zahrnují:  - Clustering: skupina dodavatelů podle kritérií jako je cena,...
    BERT F1: 0.735 | ROUGE-L R: 0.217 | ROUGE-L F1: 0.189
  ⏳ Graph RAG... 🔍 Nalezené entity pro graf: ['funkce prediktivní analytiky v plánování nákupu']
🎯 Přímý fulltext dotaz do grafu: *jaké* OR *jsou* OR *funkc* OR *predi* OR *analy* OR *pláno* OR *nákup*
  🎯 ZÁCHYT V GRAFU: Nalezen uzel [Kapitola] s názvem '15. Prediktivní analytika obchodu a logistiky | 15.3 Prediktivní analytika v plánování nákupu'
  🎯 ZÁCHYT V GRAFU: Nalezen uzel [Kapitola] s názvem '15. Prediktivní analytika obchodu a logistiky | 15.3 Prediktivní analytika v plánování nákupu | 15.3.3 Řešení prediktivní analytiky v řízení nákupu'
  🎯 ZÁCHYT V GRAFU: Nalezen uzel [Kapitola] s názvem '15

/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_5993/1325014252.py:192: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm_judge)
/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_5993/1325014252.py:193: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(embeddings)


Evaluating:   0%|          | 0/144 [00:00<?, ?it/s]


📊 VÝSLEDKY PER OTÁZKA — planovani_pa


,method,question,BERTScore_P,BERTScore_R,BERTScore_F1,ROUGE_L_P,ROUGE_L_R,ROUGE_L_F1,faithfulness,answer_relevancy,context_recall,answer_correctness
0,Naive RAG,Jaké jsou funkce prediktivní analytiky v pláno...,0.714,0.757,0.735,0.168,0.217,0.189,1.000,0.471,0.167,0.775
1,Graph RAG,Jaké jsou funkce prediktivní analytiky v pláno...,0.746,0.851,0.795,0.534,0.984,0.692,1.000,0.492,1.000,0.974
2,Hybrid GraphRAG,Jaké jsou funkce prediktivní analytiky v pláno...,0.750,0.854,0.799,0.534,0.984,0.692,1.000,0.494,1.000,0.972
3,Naive RAG,Jake jsou dimenze v řešení úloh analytiky příp...,0.694,0.694,0.694,0.236,0.232,0.234,1.000,0.518,0.929,0.439
4,Graph RAG,Jake jsou dimenze v řešení úloh analytiky příp...,0.730,0.796,0.762,0.402,0.949,0.565,1.000,0.473,1.000,0.961
5,Hybrid GraphRAG,Jake jsou dimenze v řešení úloh analytiky příp...,0.725,0.792,0.757,0.379,0.932,0.539,1.000,0.505,1.000,0.961
6,Naive RAG,Jake jsou hlavní plánovací dimenze podle jedno...,0.842,0.875,0.858,0.828,0.987,0.901,1.000,0.438,1.000,0.978
7,Graph RAG,Jake jsou hlavní plánovací dimenze podle jedno...,0.835,0.867,0.851,0.819,0.987,0.895,1.000,0.438,1.000,0.978
8,Hybrid GraphRAG,Jake jsou hlavní plánovací dimenze podle jedno...,0.746,0.807,0.775,0.808,1.000,0.894,1.000,0.431,1.000,0.953
9,Naive RAG,Jaké jsou základní cíle plánovacích úloh?,0.621,0.706,0.661,0.093,0.266,0.137,1.000,0.387,0.667,0.258



📊 PRŮMĚRNÉ SKÓRE — planovani_pa


,BERTScore_P,BERTScore_R,BERTScore_F1,ROUGE_L_P,ROUGE_L_R,ROUGE_L_F1,faithfulness,answer_relevancy,context_recall,answer_correctness
method,,,,,,,,,,
Graph RAG,0.736,0.821,0.775,0.441,0.800,0.537,0.915,0.510,0.955,0.783
Hybrid GraphRAG,0.733,0.831,0.778,0.449,0.880,0.565,0.938,0.519,0.938,0.859
Naive RAG,0.681,0.719,0.699,0.229,0.318,0.258,0.789,0.454,0.523,0.474



🏆 Nejlepší ROUGE-L Recall (úplnost):       Hybrid GraphRAG
🏆 Nejlepší BERTScore F1 (sémantika):        Hybrid GraphRAG
🏆 Nejlepší Answer Correctness (RAGAS):      Hybrid GraphRAG

✅ Výsledky uloženy → evaluation_planovani_pa.csv


,document,method,question,answer,context,time_s,ground_truth,BERTScore_P,BERTScore_R,BERTScore_F1,ROUGE_L_P,ROUGE_L_R,ROUGE_L_F1,faithfulness,answer_relevancy,context_recall,answer_correctness
0,planovani_pa,Naive RAG,Jaké jsou funkce prediktivní analytiky v pláno...,Funkce prediktivní analytiky v plánování nákup...,Účelem kapitoly je vymezit funkcionalitu predi...,18.6,- Clustering: klastry dodavatelů podle dodávan...,0.714192,0.756678,0.734821,0.167665,0.217054,0.189189,1.000000,0.470824,0.166667,0.775423
1,planovani_pa,Graph RAG,Jaké jsou funkce prediktivní analytiky v pláno...,Funkce prediktivní analytiky v plánování nákup...,### DOKUMENTY Z GRAFU:\nPrediktivní analytika ...,20.3,- Clustering: klastry dodavatelů podle dodávan...,0.745744,0.851247,0.795011,0.533613,0.984496,0.692098,1.000000,0.491682,1.000000,0.973967
2,planovani_pa,Hybrid GraphRAG,Jaké jsou funkce prediktivní analytiky v pláno...,Funkce prediktivní analytiky v plánování nákup...,### 1. DOKUMENTY Z GRAFU (PRIMÁRNÍ ZDROJ - ČTĚ...,21.5,- Clustering: klastry dodavatelů podle dodávan...,0.749811,0.854337,0.798669,0.533613,0.984496,0.692098,1.000000,0.493506,1.000000,0.971577
3,planovani_pa,Naive RAG,Jake jsou dimenze v řešení úloh analytiky příp...,Dimenze v řešení úloh analytiky přípravy a plá...,Účelem kapitoly je vymezit funkcionalitu predi...,20.7,"Čas - časová dimenze určující dobu poptávky, z...",0.693650,0.694047,0.693849,0.235632,0.231638,0.233618,1.000000,0.518056,0.928571,0.438870
4,planovani_pa,Graph RAG,Jake jsou dimenze v řešení úloh analytiky příp...,Dimenze v řešení úloh analytiky přípravy a plá...,### DOKUMENTY Z GRAFU:\n**[10.1] Dimenze v řeš...,28.0,"Čas - časová dimenze určující dobu poptávky, z...",0.729696,0.796431,0.761604,0.401914,0.949153,0.564706,1.000000,0.473013,1.000000,0.960950
5,planovani_pa,Hybrid GraphRAG,Jake jsou dimenze v řešení úloh analytiky příp...,Dimenze v řešení úloh analytiky přípravy a plá...,### 1. DOKUMENTY Z GRAFU (PRIMÁRNÍ ZDROJ - ČTĚ...,29.9,"Čas - časová dimenze určující dobu poptávky, z...",0.725155,0.791677,0.756957,0.379310,0.932203,0.539216,1.000000,0.505013,1.000000,0.961398
6,planovani_pa,Naive RAG,Jake jsou hlavní plánovací dimenze podle jedno...,Podle tabulky B-1 jsou hlavní plánovací dimenz...,Tabulka B-1: Přehled hlavních plánovacích dime...,28.1,"Základní dimenze: Časová dimenze, Regiony, Odv...",0.841777,0.874762,0.857952,0.827957,0.987179,0.900585,1.000000,0.438132,1.000000,0.978155
7,planovani_pa,Graph RAG,Jake jsou hlavní plánovací dimenze podle jedno...,Podle tabulky B-1 jsou hlavní plánovací dimenz...,### DOKUMENTY Z GRAFU:\nTabulka B-1: Přehled h...,20.9,"Základní dimenze: Časová dimenze, Regiony, Odv...",0.835250,0.867390,0.851017,0.819149,0.987179,0.895349,1.000000,0.438132,1.000000,0.978487
8,planovani_pa,Hybrid GraphRAG,Jake jsou hlavní plánovací dimenze podle jedno...,Hlavní plánovací dimenze podle jednotlivých sk...,### 1. DOKUMENTY Z GRAFU (PRIMÁRNÍ ZDROJ - ČTĚ...,27.2,"Základní dimenze: Časová dimenze, Regiony, Odv...",0.745838,0.807065,0.775245,0.808290,1.000000,0.893983,1.000000,0.431391,1.000000,0.952885
9,planovani_pa,Naive RAG,Jaké jsou základní cíle plánovacích úloh?,Plánovací úlohy mají několik základních cílů:\...,Plánování skladů a skladových zásob firmy jsou...,21.8,- Vytvořit a využít plánovací systém respektuj...,0.621298,0.706267,0.661063,0.092511,0.265823,0.137255,1.000000,0.386567,0.666667,0.257767


In [ ]:
df1 = pd.read_csv('../evaluation/evaluation_strojirenska_firma.csv')
df2 = pd.read_csv('../evaluation/evaluation_planovani_pa.csv')

metrics = ['BERTScore_F1', 'ROUGE_L_R', 'ROUGE_L_F1',
           'faithfulness', 'answer_relevancy', 'context_recall', 'answer_correctness']
metrics = [m for m in metrics if df1[m].notna().any()]

# Průměrné skóre per dokument
for label, df in [('Strojírenská firma', df1), ('Plánování PA', df2)]:
    dv = df[df['answer'] != 'TIMEOUT']
    print(f'\n=== {label} — průměry ===')
    display(dv.groupby('method')[metrics].mean().round(3))

# Souhrnné průměry přes oba dokumenty
dv_all = pd.concat([df1, df2])
dv_all = dv_all[dv_all['answer'] != 'TIMEOUT']
print('\n=== Souhrn — oba dokumenty ===')
display(dv_all.groupby('method')[metrics].mean().round(3))

# Výsledky per typ otázky — Strojírenská firma
typy_firma = {
    'Výčtové':       df1['question'].unique()[0:3],
    'Definice/účel': df1['question'].unique()[3:6],
    'Vztahové':      df1['question'].unique()[6:9],
    'Faktické':      df1['question'].unique()[9:12],
}
print('\n=== Strojírenská firma — per typ otázky ===')
for typ, qs in typy_firma.items():
    sub = df1[df1['question'].isin(qs) & (df1['answer'] != 'TIMEOUT')]
    print(f'\n{typ}:')
    display(sub.groupby('method')[['ROUGE_L_R', 'BERTScore_F1', 'answer_correctness']].mean().round(3))

# Výsledky per typ otázky — Plánování PA
typy_pa = {
    'Výčtové':       df2['question'].unique()[0:3],
    'Definice/účel': df2['question'].unique()[3:6],
    'Vztahové':      df2['question'].unique()[6:9],
    'Faktické':      df2['question'].unique()[9:12],
}
print('\n=== Plánování PA — per typ otázky ===')
for typ, qs in typy_pa.items():
    sub = df2[df2['question'].isin(qs) & (df2['answer'] != 'TIMEOUT')]
    print(f'\n{typ}:')
    display(sub.groupby('method')[['ROUGE_L_R', 'BERTScore_F1', 'answer_correctness']].mean().round(3))


=== Strojírenská firma — průměry ===


,BERTScore_F1,ROUGE_L_R,ROUGE_L_F1,faithfulness,answer_relevancy,context_recall,answer_correctness
method,,,,,,,
Graph RAG,0.748,0.792,0.372,0.861,0.629,0.833,0.689
Hybrid GraphRAG,0.798,0.862,0.527,0.917,0.596,1.000,0.756
Naive RAG,0.701,0.342,0.243,0.892,0.558,0.785,0.594



=== Plánování PA — průměry ===


,BERTScore_F1,ROUGE_L_R,ROUGE_L_F1,faithfulness,answer_relevancy,context_recall,answer_correctness
method,,,,,,,
Graph RAG,0.775,0.800,0.537,0.915,0.510,0.955,0.783
Hybrid GraphRAG,0.778,0.880,0.565,0.938,0.519,0.938,0.859
Naive RAG,0.699,0.318,0.258,0.789,0.454,0.523,0.474



=== SOUHRN — oba dokumenty ===


...

,ROUGE_L_R,BERTScore_F1,answer_correctness
method,,,
Graph RAG,0.825,0.716,0.779
Hybrid GraphRAG,0.853,0.702,0.761
Naive RAG,0.234,0.670,0.373



Vztahové:


,ROUGE_L_R,BERTScore_F1,answer_correctness
method,,,
Graph RAG,0.716,0.813,0.795
Hybrid GraphRAG,0.974,0.834,0.963
Naive RAG,0.136,0.661,0.318



Faktické:


,ROUGE_L_R,BERTScore_F1,answer_correctness
method,,,
Graph RAG,0.685,0.770,0.585
Hybrid GraphRAG,0.721,0.798,0.748
Naive RAG,0.425,0.703,0.474


In [ ]:
df1 = pd.read_csv('../evaluation/evaluation_strojirenska_firma.csv')
df2 = pd.read_csv('../evaluation/evaluation_planovani_pa.csv')
df_all = pd.concat([df1, df2])
df_all = df_all[df_all['answer'] != 'TIMEOUT']

metrics_show = ['ROUGE_L_R', 'BERTScore_F1', 'context_recall', 'answer_correctness']

# Pivot tabulka pro porovnání Answer Correctness per metoda
pivot = df_all.pivot_table(
    index=['document', 'question'],
    columns='method',
    values='answer_correctness',
    aggfunc='first'
).reset_index()
pivot.columns.name = None

# Otázky kde Naive RAG překonal Graph RAG o více než 0.4 bodů
naive_wins = pivot[
    (pivot['Naive RAG'] > pivot['Graph RAG']) &
    (pivot['Naive RAG'] - pivot['Graph RAG'] >= 0.4)
].copy()
naive_wins['diff (Naive-Graph)'] = (naive_wins['Naive RAG'] - naive_wins['Graph RAG']).round(3)
naive_wins = naive_wins.sort_values('diff (Naive-Graph)', ascending=False)

print(f"Otázky kde Naive RAG > Graph RAG (rozdíl >= 0.4, metrika: Answer Correctness): {len(naive_wins)}")
display(naive_wins[['document', 'question', 'Naive RAG', 'Graph RAG', 'Hybrid GraphRAG', 'diff (Naive-Graph)']].round(3).reset_index(drop=True))

# Detail per otázka — metriky a vygenerované odpovědi
print("\nDetail per otázka")
for _, row in naive_wins.iterrows():
    q   = row['question']
    doc = row['document']
    print(f"\n{'=' * 70}")
    print(f"Dokument: {doc} | Otázka: {q}")

    sub = df_all[df_all['question'] == q][['method'] + metrics_show].copy()
    sub = sub.rename(columns={
        'ROUGE_L_R':          'ROUGE-L Recall',
        'BERTScore_F1':       'BERTScore F1',
        'context_recall':     'Context Recall',
        'answer_correctness': 'Answer Correctness'
    }).set_index('method')
    display(sub.round(3))

    print("\nOdpovědi:")
    for method in ['Naive RAG', 'Graph RAG', 'Hybrid GraphRAG']:
        ans_row = df_all[(df_all['question'] == q) & (df_all['method'] == method)]
        if not ans_row.empty:
            answer = ans_row['answer'].values[0]
            ac     = ans_row['answer_correctness'].values[0]
            print(f"\n{method} (AC={float(ac or 0):.3f})")
            print(answer[:1000] + ('...' if len(str(answer)) > 1000 else ''))

📊 Otázky kde Naive RAG > Graph RAG (rozdíl ≥ 0.4) — metrika: Answer Correctness (celkem: 2)


,document,question,Naive RAG,Graph RAG,Hybrid GraphRAG,diff (Naive−Graph)
0,strojirenska_firma,Jaké funkce zahrnuje pokročilá analytika v mar...,0.957,0.173,0.746,0.784
1,planovani_pa,Jaké jsou funkce prediktivní analytiky v marke...,0.976,0.511,0.718,0.465



📊 Detail per otázka — metriky + odpovědi

══════════════════════════════════════════════════════════════════════
Dokument: strojirenska_firma | Otázka: Jaké funkce zahrnuje pokročilá analytika v marketingu?


,ROUGE-L Recall,BERTScore F1,Context Recall,Answer Correctness
method,,,,
Naive RAG,0.569,0.727,1.0,0.957
Graph RAG,0.176,0.640,0.0,0.173
Hybrid GraphRAG,0.451,0.711,1.0,0.746



📝 Odpovědi:

── Naive RAG (AC=0.957) ──
Pokročilá analytika v marketingu zahrnuje následující funkce:

1.  **Segmentace**: rozdělení objektů do skupin, které mají podobné charakteristiky, např. segmentace demografická, behaviorální (podle chování), geografická apod.
2.  **Analýza nákupního košíku**: zjišťuje souvislosti mezi produkty, které kupují zákazníci společně, a nabízí kombinace produktů (cross selling) apod.
3.  **Predikce odchodu zákazníků** - 'churn management': předpovídá zákazníky, kteří hodlají odejít ke konkurenci nebo přestat využívat určité produkty nebo služby.
4.  **Credit scoring**: ohodnocení zákazníka podle úvěrového rizika, zda bude splácet např. svůj úvěr.

Pokročilá analytika v marketingu se zaměřuje na prediktivní analýzu a předpovídání chování zákazníků, aby bylo možné efektivněji cílit na jejich potřeby a nabízet jim relevantní produkty nebo služby....

── Graph RAG (AC=0.173) ──
Není možné odpovědět na otázku, protože se týká marketingu, nikoli finančního ř

,ROUGE-L Recall,BERTScore F1,Context Recall,Answer Correctness
method,,,,
Naive RAG,0.983,0.840,1.0,0.976
Graph RAG,0.283,0.672,1.0,0.511
Hybrid GraphRAG,0.233,0.709,1.0,0.718



📝 Odpovědi:

── Naive RAG (AC=0.976) ──
Funkce prediktivní analytiky v marketingu zahrnují:

- Clustering:
  - klastry zákazníků strojírenské firmy podle typu výroby, příslušnosti k dodavatelskému řetězci, dislokace závodů apod.,
  - segmentace zákazníků podle technickoekonomických charakteristik,
  - segmentace marketingových akcí, podle zaměření na skupiny zákazníků, způsobu realizace, teritoriálního zaměření.

- Klasifikace :
  - klasifikace zákazníků - jejich rozřazení do stejnorodých tříd a odhad pravděpodobnosti správnosti takového zařazení pro každého zákazníka,
  - klasifikace prodávaného zboží - obvykle podle katalogu zboží,
  - klasifikace marketingových akcí - posouzení správnosti klasifikace uvedením pravděpodobnosti správného členění.

- Přiřazování podobností:
  - určování podstatných charakteristik zákazníků a na tomto základě výběr firem, které jsou obdobné našim nejlepším zákazníkům.

- Profilování, ' Popis chování ' :
  - specifikace klíčových charakteristik zákazník